# Лабораторна робота №4: Візуалізація даних 2
**Мета:** Створити інтерактивний графік гармоніки з накладеним шумом та можливістю фільтрації (IIR фільтр Баттерворта).

*Примітка: При запуску коду графік відкриється в окремому інтерактивному вікні для коректної роботи повзунків.*

In [1]:
%matplotlib tk
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, Button, CheckButtons
from scipy.signal import butter, filtfilt
import warnings


# --- 1. НАЛАШТУВАННЯ ДАНИХ ТА ШУМУ ---
# Задаємо фіксований "seed", щоб шум не генерувався наново при кожному русі слайдера
# Це виконує умову: "якщо ви змінили параметри гармоніки... шум має залишитись таким як і був"
np.random.seed(42)
t = np.linspace(0, 10, 1000)
fs = 100.0  # Частота дискретизації (1000 точок / 10 сек)
base_noise = np.random.normal(0, 1, len(t)) # Базовий масив шуму N(0,1)

# Початкові параметри
init_amp = 1.0
init_freq = 1.0
init_phase = 0.0
init_n_mean = 0.0
init_n_cov = 0.1
init_cutoff = 2.0  # Частота зрізу для фільтра (Гц)

# --- 2. ФУНКЦІЇ ЛОГІКИ ---
def harmonic_with_noise(amplitude, frequency, phase, noise_mean, noise_covariance, show_noise):
    """
    Генерує чисту гармоніку та додає до неї шум.
    Дисперсія (covariance) визначає розкид. Множимо базовий шум на корінь з дисперсії.
    """
    omega = 2 * np.pi * frequency
    clean_signal = amplitude * np.sin(omega * t + phase)
    noise = np.sqrt(noise_covariance) * base_noise + noise_mean
    noisy_signal = clean_signal + noise
    
    # Завдання вимагає параметр show_noise, тому повертаємо потрібне
    result_signal = noisy_signal if show_noise else clean_signal
    return clean_signal, noisy_signal, result_signal

def butter_lowpass_filter(data, cutoff, fs, order=4):
    """Реалізація низькочастотного IIR фільтру Баттерворта"""
    nyq = 0.5 * fs # Частота Найквіста
    normal_cutoff = cutoff / nyq
    # Захист від помилок при екстремальних значеннях слайдера
    if normal_cutoff >= 1: normal_cutoff = 0.99
    if normal_cutoff <= 0: normal_cutoff = 0.01
    
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    # filtfilt застосовує фільтр вперед і назад для уникнення фазового зсуву
    y = filtfilt(b, a, data) 
    return y

# --- 3. НАЛАШТУВАННЯ ІНТЕРФЕЙСУ ---
fig, ax = plt.subplots(figsize=(12, 8))
plt.subplots_adjust(left=0.1, bottom=0.45) # Залишаємо місце знизу для слайдерів

# Отримуємо початкові дані
clean, noisy, _ = harmonic_with_noise(init_amp, init_freq, init_phase, init_n_mean, init_n_cov, True)
filtered = butter_lowpass_filter(noisy, init_cutoff, fs)

# Малюємо лінії
l_clean, = ax.plot(t, clean, lw=2, color='green', linestyle='--', label='Чиста (еталон)')
l_noisy, = ax.plot(t, noisy, lw=1, color='red', alpha=0.5, label='Зашумлена')
l_filtered, = ax.plot(t, filtered, lw=2.5, color='blue', label='Відфільтрована')

ax.legend(loc='upper right')
ax.set_title('Інтерактивна візуалізація гармоніки: Шум та Фільтрація', fontsize=14)
ax.set_xlabel('Час (секунди)')
ax.set_ylabel('Амплітуда')
ax.grid(True)

# --- 4. СЛАЙДЕРИ (SLIDERS) ---
axcolor = 'lightgoldenrodyellow'
ax_amp    = plt.axes([0.15, 0.35, 0.60, 0.03], facecolor=axcolor)
ax_freq   = plt.axes([0.15, 0.30, 0.60, 0.03], facecolor=axcolor)
ax_phase  = plt.axes([0.15, 0.25, 0.60, 0.03], facecolor=axcolor)
ax_n_mean = plt.axes([0.15, 0.20, 0.60, 0.03], facecolor=axcolor)
ax_n_cov  = plt.axes([0.15, 0.15, 0.60, 0.03], facecolor=axcolor)
ax_cutoff = plt.axes([0.15, 0.10, 0.60, 0.03], facecolor=axcolor)

s_amp    = Slider(ax_amp, 'Амплітуда', 0.1, 10.0, valinit=init_amp)
s_freq   = Slider(ax_freq, 'Частота', 0.1, 10.0, valinit=init_freq)
s_phase  = Slider(ax_phase, 'Фаза', 0.0, 2*np.pi, valinit=init_phase)
s_n_mean = Slider(ax_n_mean, 'Шум Mean', -2.0, 2.0, valinit=init_n_mean)
s_n_cov  = Slider(ax_n_cov, 'Шум Дисперсія', 0.0, 5.0, valinit=init_n_cov)
s_cutoff = Slider(ax_cutoff, 'Зріз Фільтра', 0.1, 20.0, valinit=init_cutoff)

# Функція оновлення при русі слайдерів
def update(val):
    c, n, _ = harmonic_with_noise(s_amp.val, s_freq.val, s_phase.val, s_n_mean.val, s_n_cov.val, True)
    f = butter_lowpass_filter(n, s_cutoff.val, fs)
    
    l_clean.set_ydata(c)
    l_noisy.set_ydata(n)
    l_filtered.set_ydata(f)
    
    # Динамічне масштабування осей
    ax.relim()
    ax.autoscale_view()
    fig.canvas.draw_idle()

# Підключаємо слайдери до функції оновлення
for slider in [s_amp, s_freq, s_phase, s_n_mean, s_n_cov, s_cutoff]:
    slider.on_changed(update)

# --- 5. КНОПКА RESET ---
resetax = plt.axes([0.8, 0.025, 0.1, 0.04])
button = Button(resetax, 'Reset', color=axcolor, hovercolor='0.975')

def reset(event):
    for slider in [s_amp, s_freq, s_phase, s_n_mean, s_n_cov, s_cutoff]:
        slider.reset()
button.on_clicked(reset)

# --- 6. ЧЕКБОКСИ (ВІДОБРАЖЕННЯ) ---
rax = plt.axes([0.8, 0.2, 0.15, 0.15], facecolor=axcolor)
lines = [l_noisy, l_filtered, l_clean]
labels = [l.get_label() for l in lines]
visibility = [l.get_visible() for l in lines]
check = CheckButtons(rax, labels, visibility)

def toggle_visibility(label):
    index = labels.index(label)
    lines[index].set_visible(not lines[index].get_visible())
    fig.canvas.draw_idle()
check.on_clicked(toggle_visibility)

# Інструкція для користувача виводиться в консоль при запуску
print("="*50)
print("ІНСТРУКЦІЯ КОРИСТУВАЧА:")
print("1. Використовуйте нижні повзунки (слайдери) для зміни параметрів:")
print("   - Амплітуда, Частота, Фаза: змінюють саму гармоніку.")
print("   - Шум Mean, Шум Дисперсія: змінюють накладений шум.")
print("   - Зріз Фільтра: регулює, наскільки сильно фільтр 'згладжує' шум.")
print("2. Використовуйте чекбокси праворуч, щоб вмикати/вимикати відображення")
print("   чистої, зашумленої або відфільтрованої гармоніки.")
print("3. Натисніть кнопку 'Reset', щоб повернути всі значення до початкових.")
print("="*50)

plt.show()

ІНСТРУКЦІЯ КОРИСТУВАЧА:
1. Використовуйте нижні повзунки (слайдери) для зміни параметрів:
   - Амплітуда, Частота, Фаза: змінюють саму гармоніку.
   - Шум Mean, Шум Дисперсія: змінюють накладений шум.
   - Зріз Фільтра: регулює, наскільки сильно фільтр 'згладжує' шум.
2. Використовуйте чекбокси праворуч, щоб вмикати/вимикати відображення
   чистої, зашумленої або відфільтрованої гармоніки.
3. Натисніть кнопку 'Reset', щоб повернути всі значення до початкових.
